# IAM LLM fine-tuned inference — **vLLM** (Qwen3.5-9B)

Runs a trained LoRA adapter on an SFT split and writes the prediction CSV in
rag_core's format. Continuous batching + paged KV + built-in JSON-schema
structured output: a whole set in one `llm.generate` call.

**Use a fresh runtime** (not the training one): vLLM pins its own torch /
transformers and must not be mixed with the unsloth training pins.

The adapter is **merged into bf16 weights first** (generic PEFT merge, verified by
checking a weight actually changed) and vLLM serves the merged model. This avoids
vLLM's Qwen3.5 LoRA-module mapping bug (issue #38085). The merged copy lives on local disk (`/content/merged/<TAG>_<ckpt>`,
~16-18 GB) and is rebuilt per checkpoint - never written to Drive.

The reported runs used vLLM 0.28.0 and transformers 5.15.1 on an A100 (80 GB).

Runtime → Change runtime type → **A100 / H100**.


In [ ]:
# vLLM pulls in its own torch (CUDA 13.x wheels). Colab's preinstalled torchaudio is
# built against CUDA 12.8 (breaks `import transformers`) and its torchao 0.10 is below
# peft's minimum (0.16) - neither is needed here, so remove both. The cuml / dask-cuda "incompatible" lines pip prints are Colab's RAPIDS
# packages complaining; harmless.
!pip install -q vllm peft
!pip uninstall -y -q torchaudio torchao   # Colab preinstalls old versions of both; neither is needed
import vllm, transformers, peft, torch
print("vllm", vllm.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
# ── which run / which checkpoint / which split ──────────────────────────────
os.environ["SFT_BASE_MODEL"] = "Qwen/Qwen3.5-9B"
os.environ["SFT_LR"]         = "5e-4"               # must match the training run -> TAG
os.environ["SFT_LORA_R"]     = "16"
os.environ["SFT_SEED"]       = "42"
os.environ["SFT_SFX"]        = ""                   # "" | "_nonb"
os.environ["SFT_SPLIT"]      = "val_infer"          # val_infer | test | test_sub01 .. test_sub05
os.environ["SFT_LIMIT"]      = "0"                  # 0 = all (REPORT); N = probe
os.environ["SFT_CONSTRAIN"]  = "1"                  # 1 = schema-constrained (REPORT); 0 = free
# default adapter = adapters/<TAG> (best eval_loss). A specific step instead:
# os.environ["SFT_ADAPTER"] = "/content/drive/MyDrive/finetune/checkpoints/<TAG>/checkpoint-600"
os.environ.pop("SFT_ADAPTER", None)


In [ ]:
# `!python` (a real subprocess), NOT `%run`: vLLM's engine calls sys.stdout.fileno(),
# which the notebook kernel's stdout does not have (UnsupportedOperation: fileno).
# The subprocess inherits the os.environ set in the cell above.
!python /content/drive/MyDrive/finetune/colab_infer_vllm.py


To score another checkpoint of the same run, set `SFT_ADAPTER` in the cell above
and re-run the cell below it (the merge is cached per checkpoint; the vLLM engine
is rebuilt). To switch split, change `SFT_SPLIT` and re-run.


## Evaluate back on your machine (from the project root)
Sync the CSV from `MyDrive/finetune/preds/`, then either the quick table
```bash
python3 finetune/score_preds.py --gt make_datasets/output/split/AR6_R10sd_val10_gt.xlsx \
    --pred <csv1> <csv2> ... --label 2e-4 5e-4 1e-3
```
or the full rag_core report (same multi-sheet xlsx as the RAG runs; lands next to the CSV as `<name>_eval.xlsx`):
```bash
IAM_LLM_PRED_PATH=<csv> \
IAM_LLM_BATCH_SHEET_PATH=make_datasets/output/split/AR6_R10sd_val10_list.xlsx \
IAM_LLM_GT_PATH=make_datasets/output/split/AR6_R10sd_val10_gt.xlsx \
python3 -c "import sys; sys.path.insert(0, '.'); import rag_core; rag_core.run_evaluation()"
```
(test / sub01..05: swap the list + gt paths.)
